# Kultura 2012 Survey — Full Anonymization Workflow with Amnesia

This notebook demonstrates end-to-end anonymization of the Kultura 2012 cultural survey data using the Amnesia REST API.

**Workflow:**
1. Load full SPSS survey data and save as Parquet (preserving all 275 columns)
2. Extract quasi-identifiers for Amnesia anonymization
3. Start Amnesia server, load QI data, generate hierarchies
4. Run k-anonymity anonymization
5. Merge anonymized QI columns back with full dataset
6. Compare original vs anonymized full dataset

## 1. Setup — Imports and Paths

In [110]:
import subprocess
import time
import json
import requests
import pandas as pd
import pyreadstat
from pathlib import Path

# Paths
JAR_PATH = Path("/home/jupyter-vojta/notebooks/Amnesia/target/amnesiaBackEnd-1.0-SNAPSHOT.jar")
AMNESIA_DIR = JAR_PATH.parent.parent
BASE_URL = "http://localhost:8181"

# Data paths
DATA_DIR = Path("../data/culture_dataverse_files")
OUTPUT_DIR = Path("../data")

print("Imports OK, paths set up.")

Imports OK, paths set up.


## 2. Load Full Dataset and Save as Parquet

In [111]:
# SPSS file path
SPSS_FILE = DATA_DIR / "Kult2012_3kraje_UstVysZli_CSDA_pub_nove_bez_jmen.sav"

# Read SPSS file
df, meta = pyreadstat.read_sav(SPSS_FILE)

print(f"Loaded {df.shape[0]} rows × {df.shape[1]} columns")
print(f"Created: {meta.creation_time}")

# Save full dataset as Parquet (preserves all columns and types)
parquet_path = OUTPUT_DIR / "culture_full.parquet"
df.to_parquet(parquet_path, index=False)
print(f"\nSaved full dataset to: {parquet_path}")
print(f"Parquet file size: {parquet_path.stat().st_size / 1024 / 1024:.2f} MB")

Loaded 3679 rows × 275 columns
Created: 2015-01-07 11:54:08

Saved full dataset to: ../data/culture_full.parquet
Parquet file size: 0.75 MB


## 3. Identify Quasi-Identifiers for Anonymization

In [112]:
# Key demographic/quasi-identifier columns for anonymization
# Note: pohlavi (gender) excluded - only 2 values, Amnesia can't generate range hierarchy
QI_COLUMNS = [
    "KRAJ",       # Region (3 categories)
    "vek",        # Age in years
    "t_vzd",      # Education level (4 kat)
    "velobce3b",  # Town size category (3 kat)
]

# Check which columns exist
available_qis = [col for col in QI_COLUMNS if col in df.columns]
print("Available quasi-identifiers for anonymization:")
for col in available_qis:
    label = meta.column_names_to_labels.get(col, "(no label)")
    print(f"  {col}: {label}")

# Create subset with just quasi-identifiers for Amnesia
df_qi = df[available_qis].copy()
print(f"\nSubset shape: {df_qi.shape}")

# Export QI subset as CSV for Amnesia
qi_csv = OUTPUT_DIR / "culture_qi.csv"
df_qi.to_csv(qi_csv, index=False)
print(f"\nExported QI data to: {qi_csv}")

Available quasi-identifiers for anonymization:
  KRAJ: Kraj
  vek: Věk
  t_vzd: t_vzd - Transformované vzdělání (4 kat. VŠ vč. VŠO)
  velobce3b: Velikost obce (dle pocetob_20130101)

Subset shape: (3679, 4)

Exported QI data to: ../data/culture_qi.csv


In [113]:
# Quick look at value distributions for each quasi-identifier
for col in available_qis:
    print(f"\n{col} ({meta.column_names_to_labels.get(col, '(no label)')}):")
    print(df_qi[col].value_counts().sort_index())


KRAJ (Kraj):
KRAJ
1.0    1255
2.0    1204
3.0    1220
Name: count, dtype: int64

vek (Věk):
vek
20.0    97
21.0    91
22.0    80
23.0    80
24.0    72
        ..
85.0     6
86.0     1
87.0     2
88.0     4
92.0     1
Name: count, Length: 70, dtype: int64

t_vzd (t_vzd - Transformované vzdělání (4 kat. VŠ vč. VŠO)):
t_vzd
1.0     667
2.0    1555
3.0    1128
4.0     319
Name: count, dtype: int64

velobce3b (Velikost obce (dle pocetob_20130101)):
velobce3b
1.0     934
2.0     989
3.0    1739
Name: count, dtype: int64


## 4. Start Amnesia Server

In [114]:
# Start the Amnesia backend server
proc = subprocess.Popen(
    [
        "java", "-Xms1024m", "-Xmx4096m",
        "-Dorg.eclipse.jetty.server.Request.maxFormKeys=1000000",
        "-Dorg.eclipse.jetty.server.Request.maxFormContentSize=1000000",
        "-jar", str(JAR_PATH),
        "--server.port=8181",
    ],
    cwd=str(AMNESIA_DIR),
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
)

print("Waiting for server to start...")
for _ in range(20):
    time.sleep(2)
    try:
        r = requests.post(f"{BASE_URL}/getSession", timeout=3)
        if r.status_code == 200:
            print("Server is up!")
            break
    except requests.exceptions.ConnectionError:
        pass
else:
    raise RuntimeError("Server did not start in time")

Waiting for server to start...
Server is up!


## 5. Get Session and Load QI Data

In [115]:
# Get a session
r = requests.post(f"{BASE_URL}/getSession")
session_id = r.json()["Session_Id"]
headers = {"Cookie": f"JSESSIONID={session_id}"}
print(f"Session ID: {session_id}")

Session ID: node046pptgje7cna4a9a2ao9tk9r1


In [116]:
# Define column types for the QI data
column_types = json.dumps({col: "int" for col in available_qis})

# Load the CSV file
with open(qi_csv, "rb") as f:
    r = requests.post(
        f"{BASE_URL}/loadData",
        headers=headers,
        files={"file": ("culture_qi.csv", f, "text/csv")},
        data={
            "del": ",",
            "datasetType": "tabular",
            "columnsType": column_types,
        },
    )

print(r.json())

{'Status': 'Success', 'Message': 'Dataset is  successfully loaded!'}


## 6. Generate Hierarchies on Server

In [117]:
# Generate hierarchy for 'vek' (age) - range-based
r = requests.post(
    f"{BASE_URL}/generateHierarchy",
    headers=headers,
    data={
        "hierType": "range",
        "varType": "int",
        "attribute": "vek",
        "hierName": "vek_hier",
        "startLimit": 20,
        "endLimit": 100,
        "fanout": 3,
        "step": 10,
    },
)
print("Age hierarchy generated:", r.json())

Age hierarchy generated: {'hierType': 'range', 'name': 'vek_hier', 'type': 'int', 'level1': {'20-100': ['20-50', '50-80', '80-100', '(null)']}, 'height': 3, 'level2': {'80-100': ['80-90', '90-100'], '50-80': ['50-60', '60-70', '70-80'], '20-50': ['20-30', '30-40', '40-50']}}


In [118]:
# Generate hierarchy for 'KRAJ' (region) - range-based with small fanout
r = requests.post(
    f"{BASE_URL}/generateHierarchy",
    headers=headers,
    data={
        "hierType": "range",
        "varType": "int",
        "attribute": "KRAJ",
        "hierName": "kraj_hier",
        "startLimit": 1,
        "endLimit": 3,
        "fanout": 2,
        "step": 1,
    },
)
print("Region hierarchy generated:", r.json())

Region hierarchy generated: {'hierType': 'range', 'name': 'kraj_hier', 'type': 'int', 'level1': {'1-3': ['1-2', '2-3', '(null)']}, 'height': 2}


In [119]:
# Skip gender - too few values for range hierarchy. Use it as-is or drop it.
print("Skipping pohlavi (gender) - only 2 values, range hierarchy doesn't work well")
print("Will exclude from anonymization bind below")

Skipping pohlavi (gender) - only 2 values, range hierarchy doesn't work well
Will exclude from anonymization bind below


In [120]:
# Generate hierarchy for 't_vzd' (education) - range-based
r = requests.post(
    f"{BASE_URL}/generateHierarchy",
    headers=headers,
    data={
        "hierType": "range",
        "varType": "int",
        "attribute": "t_vzd",
        "hierName": "tvzd_hier",
        "startLimit": 1,
        "endLimit": 4,
        "fanout": 2,
        "step": 1,
    },
)
print("Education hierarchy generated:", r.json())

Education hierarchy generated: {'hierType': 'range', 'name': 'tvzd_hier', 'type': 'int', 'level1': {'1-4': ['1-2', '2-3', '3-4', '(null)']}, 'height': 2}


In [121]:
# Generate hierarchy for 'velobce3b' (town size) - range-based
r = requests.post(
    f"{BASE_URL}/generateHierarchy",
    headers=headers,
    data={
        "hierType": "range",
        "varType": "int",
        "attribute": "velobce3b",
        "hierName": "velobce_hier",
        "startLimit": 1,
        "endLimit": 3,
        "fanout": 2,
        "step": 1,
    },
)
print("Town size hierarchy generated:", r.json())

Town size hierarchy generated: {'hierType': 'range', 'name': 'velobce_hier', 'type': 'int', 'level1': {'1-3': ['1-2', '2-3', '(null)']}, 'height': 2}


## 7. Run Anonymization

In [122]:
# Bind only attributes with working hierarchies (skip pohlavi/gender)
bind = json.dumps({
    "vek": "vek_hier",
    "KRAJ": "kraj_hier",
    "t_vzd": "tvzd_hier",
    "velobce3b": "velobce_hier",
})

k_value = 5
r = requests.post(
    f"{BASE_URL}/anonymization",
    headers=headers,
    data={"bind": bind, "k": k_value},
)

result = r.json()
print("Anonymization response:", result)

if "Solutions" in result:
    solutions = result["Solutions"]
    print(f"\nFound {len(solutions)} solutions:\n")
    for name, info in sorted(solutions.items()):
        print(f"  {name}: levels={info['levels']}, result={info['result']}")
else:
    print(f"Error: {result.get('Message', 'Unknown error')}")
    solutions = {}

Anonymization response: {'Solutions': {'sol80': {'result': 'unsafe', 'levels': '[2,2,0,2]'}, 'sol81': {'result': 'unsafe', 'levels': '[2,1,2,1]'}, 'sol82': {'result': 'unsafe', 'levels': '[2,1,1,2]'}, 'sol83': {'result': 'unsafe', 'levels': '[2,0,2,2]'}, 'sol84': {'result': 'safe', 'levels': '[1,3,2,0]'}, 'sol85': {'result': 'unsafe', 'levels': '[1,3,1,1]'}, 'sol86': {'result': 'safe', 'levels': '[1,3,0,2]'}, 'sol87': {'result': 'unsafe', 'levels': '[1,2,2,1]'}, 'sol88': {'result': 'unsafe', 'levels': '[1,2,1,2]'}, 'sol89': {'result': 'unsafe', 'levels': '[1,1,2,2]'}, 'sol91': {'result': 'unsafe', 'levels': '[0,3,1,2]'}, 'sol92': {'result': 'safe', 'levels': '[0,2,2,2]'}, 'sol93': {'result': 'safe', 'levels': '[2,3,2,0]'}, 'sol94': {'result': 'unsafe', 'levels': '[2,3,1,1]'}, 'sol95': {'result': 'safe', 'levels': '[2,3,0,2]'}, 'sol96': {'result': 'safe', 'levels': '[2,2,2,1]'}, 'sol97': {'result': 'unsafe', 'levels': '[2,2,1,2]'}, 'sol10': {'result': 'unsafe', 'levels': '[0,1,1,0]'}, '

## 8. Download Anonymized Solution

In [127]:
# Pick the first safe solution and download it
safe = [(name, info) for name, info in solutions.items() if info["result"] == "safe"]

if safe:
    chosen_name, chosen_info = safe[0]
    print(f"Using {chosen_name}: levels={chosen_info['levels']}\n")

    # Download anonymized data
    r = requests.post(
        f"{BASE_URL}/getSolution",
        headers=headers,
        data={"sol": chosen_info["levels"]},
    )

    anon_qi_path = OUTPUT_DIR / "culture_anon_qi.csv"
    anon_qi_path.write_bytes(r.content)
    print(f"Saved anonymized QI to: {anon_qi_path}")
    
    # Show anonymized data
    anon_qi_df = pd.read_csv(anon_qi_path)
    print("\nAnonymized QI data (first 10 rows):")
    print(anon_qi_df.head(10))
else:
    print("No safe solutions found for k=5. Try a lower k value.")
    anon_qi_path = None

Using sol84: levels=[1,3,2,0]

Saved anonymized QI to: ../data/culture_anon_qi.csv

Anonymized QI data (first 10 rows):
  KRAJ     vek t_vzd  velobce3b
0  2-3  20-100   1-4        2.0
1  2-3  20-100   1-4        3.0
2  1-2  20-100   1-4        3.0
3  2-3  20-100   1-4        2.0
4  1-2  20-100   1-4        2.0
5  1-2  20-100   1-4        2.0
6  1-2  20-100   1-4        3.0
7  2-3  20-100   1-4        1.0
8  1-2  20-100   1-4        3.0
9  2-3  20-100   1-4        1.0


## 9. Merge Anonymized QI Back with Full Dataset

In [128]:
# Load original full dataset and anonymized QI
df_original = pd.read_parquet(parquet_path)
df_anon_qi = pd.read_csv(anon_qi_path)

# Add row index to both for merging
df_original = df_original.reset_index(drop=True)
df_anon_qi = df_anon_qi.reset_index(drop=True)

# Replace QI columns in original with anonymized versions
df_anonymized = df_original.copy()
for col in available_qis:
    df_anonymized[col] = df_anon_qi[col]

print(f"Original shape: {df_original.shape}")
print(f"Anonymized shape: {df_anonymized.shape}")

# Save anonymized full dataset
anon_parquet_path = OUTPUT_DIR / "culture_anonymized.parquet"
df_anonymized.to_parquet(anon_parquet_path, index=False)
print(f"\nSaved anonymized full dataset to: {anon_parquet_path}")
print(f"File size: {anon_parquet_path.stat().st_size / 1024 / 1024:.2f} MB")

Original shape: (3679, 275)
Anonymized shape: (3679, 275)

Saved anonymized full dataset to: ../data/culture_anonymized.parquet
File size: 0.75 MB


## 10. Compare Original vs Anonymized

In [129]:
# Compare original vs anonymized for QI columns
print("=== First 10 rows comparison ===")
compare_df = pd.concat(
    [df_original[available_qis].head(10), df_anonymized[available_qis].head(10)],
    axis=1,
    keys=["Original", "Anonymized"]
)
print(compare_df)

=== First 10 rows comparison ===
  Original                       Anonymized                        
      KRAJ   vek t_vzd velobce3b       KRAJ     vek t_vzd velobce3b
0      1.0  57.0   2.0       1.0        2-3  20-100   1-4       2.0
1      1.0  38.0   2.0       1.0        2-3  20-100   1-4       3.0
2      1.0  21.0   3.0       1.0        1-2  20-100   1-4       3.0
3      1.0  49.0   2.0       1.0        2-3  20-100   1-4       2.0
4      1.0  60.0   2.0       1.0        1-2  20-100   1-4       2.0
5      1.0  37.0   3.0       1.0        1-2  20-100   1-4       2.0
6      1.0  69.0   1.0       1.0        1-2  20-100   1-4       3.0
7      1.0  64.0   2.0       1.0        2-3  20-100   1-4       1.0
8      1.0  21.0   3.0       1.0        1-2  20-100   1-4       3.0
9      1.0  49.0   1.0       1.0        2-3  20-100   1-4       1.0


## 11. Verify k-Anonymity

In [130]:
# Compare original vs anonymized for QI columns
print("=== First 10 rows comparison ===")
compare_df = pd.concat(
    [df_original[available_qis].head(10), df_anonymized[available_qis].head(10)],
    axis=1,
    keys=["Original", "Anonymized"]
)
print(compare_df)

=== First 10 rows comparison ===
  Original                       Anonymized                        
      KRAJ   vek t_vzd velobce3b       KRAJ     vek t_vzd velobce3b
0      1.0  57.0   2.0       1.0        2-3  20-100   1-4       2.0
1      1.0  38.0   2.0       1.0        2-3  20-100   1-4       3.0
2      1.0  21.0   3.0       1.0        1-2  20-100   1-4       3.0
3      1.0  49.0   2.0       1.0        2-3  20-100   1-4       2.0
4      1.0  60.0   2.0       1.0        1-2  20-100   1-4       2.0
5      1.0  37.0   3.0       1.0        1-2  20-100   1-4       2.0
6      1.0  69.0   1.0       1.0        1-2  20-100   1-4       3.0
7      1.0  64.0   2.0       1.0        2-3  20-100   1-4       1.0
8      1.0  21.0   3.0       1.0        1-2  20-100   1-4       3.0
9      1.0  49.0   1.0       1.0        2-3  20-100   1-4       1.0


In [131]:
# Verify k-anonymity: check minimum group sizes in anonymized data
group_sizes = df_anonymized.groupby(available_qis).size()
print(f"\nMinimum group size: {group_sizes.min()}")
print(f"Maximum group size: {group_sizes.max()}")
print(f"Number of unique groups: {len(group_sizes)}")
print(f"\nDistribution of group sizes:")
print(group_sizes.value_counts().sort_index().head(20))


Minimum group size: 199
Maximum group size: 977
Number of unique groups: 6

Distribution of group sizes:
199    1
294    1
695    1
735    1
762    1
977    1
Name: count, dtype: int64


## 12. Clean Up

In [132]:
# Clear the session on the server
r = requests.post(f"{BASE_URL}/clearSession", headers=headers)
print("Session cleared:", r.json())

# Stop the server process
proc.terminate()
print("Server stopped.")

Session cleared: {'Status': 'Success', 'Message': 'Session is cleared!'}
Server stopped.
